# GeoPilot TCC — processamento altimétrico no Google Colab

Este notebook reproduz o processamento automatizado utilizado no TCC, sem exigir a instalação local do ambiente Python. Ele utiliza o script oficial do repositório e gera o perfil altimétrico, os pontos amostrados e o resumo da rota.

**Entradas:** uma rota em GeoPackage (`.gpkg`) e um Modelo Digital de Elevação (`.tif`). Para arquivos grandes, recomenda-se armazená-los no Google Drive.

## 1. Obter o código e instalar as dependências

In [ ]:
!git clone -q https://github.com/willigfreitas/geopilot-tcc-altimetria.git
%cd geopilot-tcc-altimetria
!python -m pip install -q -r requirements.txt
print("Ambiente preparado com sucesso.")

## 2. Selecionar a forma de acesso aos dados

Escolha **uma** das opções abaixo. Para um MDE grande, use o Google Drive.

### Opção A — enviar os arquivos pelo navegador

In [ ]:
from google.colab import files

print("Selecione a rota em formato GPKG.")
rota_enviada = files.upload()
ROTA = next(iter(rota_enviada))

print("Selecione o MDE em formato TIF.")
mde_enviado = files.upload()
MDE = next(iter(mde_enviado))

print(f"Rota: {ROTA}")
print(f"MDE: {MDE}")

### Opção B — utilizar arquivos armazenados no Google Drive

Execute esta célula no lugar da Opção A e ajuste os dois caminhos.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# ROTA = '/content/drive/MyDrive/GeoPilot/dados/rota.gpkg'
# MDE = '/content/drive/MyDrive/GeoPilot/dados/mde.tif'
# print(f"Rota: {ROTA}")
# print(f"MDE: {MDE}")

## 3. Definir os parâmetros e executar

In [ ]:
from pathlib import Path
import subprocess
import sys

SRC_CALCULO = 'EPSG:31983'
ESPACAMENTO_M = 30.0
SAIDA = Path('/content/geopilot_resultado')

comando = [
    sys.executable,
    'tools/geopilot_geo_route.py',
    '--routes', str(ROTA),
    '--dem', str(MDE),
    '--output', str(SAIDA),
    '--crs', SRC_CALCULO,
    '--sample-spacing', str(ESPACAMENTO_M),
]
subprocess.run(comando, check=True)

## 4. Conferir os resultados

In [ ]:
import json
import pandas as pd
from IPython.display import display

resumo = json.loads((SAIDA / 'resumo_rota.json').read_text(encoding='utf-8'))
perfil = pd.read_csv(SAIDA / 'perfil_altimetrico.csv')

display(pd.DataFrame({
    'Indicador': [
        'Distância (km)', 'Amostras', 'Cota inicial (m)',
        'Cota final (m)', 'Subida acumulada (m)',
        'Descida acumulada (m)', 'Declividade absoluta média (%)'
    ],
    'Valor': [
        resumo['distance_km'], resumo['sample_count'],
        resumo['initial_elevation_m'], resumo['final_elevation_m'],
        resumo['accumulated_ascent_m'], resumo['accumulated_descent_m'],
        resumo['mean_absolute_grade_percent']
    ]
}))
display(perfil.head())

## 5. Baixar as saídas

In [ ]:
import shutil
from google.colab import files

arquivo_zip = shutil.make_archive('/content/geopilot_resultado', 'zip', SAIDA)
files.download(arquivo_zip)

## Observações de reprodutibilidade

- O ambiente temporário do Colab é apagado ao encerrar a sessão; baixe as saídas ou copie-as para o Google Drive.
- O resultado depende do mesmo arquivo de rota, do mesmo MDE, do SRC de cálculo e do espaçamento de amostragem.
- O notebook não substitui o script: ele funciona como uma interface para executar a versão testada e versionada no GitHub.